# The Covariance Matrix

Datasets rarely stop at two variables. With five variables there are five variances and ten pairwise covariances — and the tidy way to organize all of them is the **covariance matrix** $\Sigma$: variances on the main diagonal, covariances everywhere else. One object holds the complete second-order description of a multivariate dataset, which is why $\Sigma$ is ubiquitous in statistics and machine learning. It is based on the eighth video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

We have collected quite a menagerie of variances and covariances: the children's age against height, grades, and naps; the three games with covariances $+1$, $-1$, $0$; the call center's $-7.878$. Each was a single number about a single pair. Now imagine a large dataset with five variables — the variances of each one *and* the covariances of every pair (ten of them). We need a container, and the container is a matrix.

After completing this notebook you will be able to:

- Assemble the covariance matrix of any dataset: $\Sigma_{ii} = \mathrm{Var}(X_i)$ on the diagonal, $\Sigma_{ij} = \mathrm{Cov}(X_i, X_j)$ off it.
- Read the matrix's built-in structure: symmetry, variances on the diagonal.
- Compute $\Sigma$ with `np.cov` and by hand, for data and for distributions.
- Recognize where covariance matrices appear in statistics and machine learning.

## 2. Theory

### Two variables: the smallest covariance matrix

For a pair $(X, Y)$, arrange the four second-order quantities in a $2\times2$ grid — row and column labeled by the variables, each cell holding the covariance of its row variable with its column variable:

$$
\Sigma =
\begin{bmatrix}
\mathrm{Var}(X) & \mathrm{Cov}(X, Y) \\
\mathrm{Cov}(X, Y) & \mathrm{Var}(Y)
\end{bmatrix}
$$

The **variances sit on the main diagonal** (a variable's covariance with itself *is* its variance) and the **covariances sit off the diagonal**. For the call-center data ($\mathrm{Var}(X) = 8.526$, $\mathrm{Var}(Y) = 10.163$, $\mathrm{Cov} = -7.878$):

$$
\Sigma =
\begin{bmatrix}
8.526 & -7.878 \\
-7.878 & 10.163
\end{bmatrix}
$$

### The games

Games 1 and 2 (each player: variance 1) give the two matrices

$$
\Sigma_{\text{game 1}} = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}
\qquad\qquad
\Sigma_{\text{game 2}} = \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}
$$

— identical diagonals, opposite corners: the matrices differ exactly where the games differ.

### More variables, same pattern

For three variables $(X, Y, Z)$, a $3\times3$ matrix: variances $\mathrm{Var}(X), \mathrm{Var}(Y), \mathrm{Var}(Z)$ on the main diagonal, and in cell $(X, Y)$ the covariance $\mathrm{Cov}(X, Y)$, in $(X, Z)$ the covariance $\mathrm{Cov}(X, Z)$, and so on. For five variables $(V, W, X, Y, Z)$: a $5\times5$ matrix — 5 variances on the diagonal and the 10 pairwise covariances filling both triangles.

$$
\Sigma_{ij} = \mathrm{Cov}(X_i, X_j) \qquad \big(\text{so } \Sigma_{ii} = \mathrm{Var}(X_i)\big)
$$

Two structural facts come for free:

- **Symmetry:** $\mathrm{Cov}(X_i, X_j) = \mathrm{Cov}(X_j, X_i)$, so $\Sigma$ equals its own transpose — each covariance appears twice, mirrored across the diagonal.
- **Diagonal = variances:** setting $j = i$ recovers $\mathrm{Cov}(X_i, X_i) = \mathrm{Var}(X_i)$.

This matrix is called **$\Sigma$, the covariance matrix**, and it is very useful in statistics and machine learning — it parameterizes multivariate normal distributions, drives principal component analysis, and appears in portfolio risk, Kalman filters, and Gaussian processes.

## 3. Worked Examples

### Example 1: the children's age and height

With the population convention ($\div n$): $\mathrm{Var}(\text{age}) = 8.25$, $\mathrm{Var}(\text{height}) = 35.6$, $\mathrm{Cov} = 17$:

$$
\Sigma = \begin{bmatrix} 8.25 & 17 \\ 17 & 35.6 \end{bmatrix}
$$

### Example 2: reading a 3×3 matrix

Suppose $\Sigma = \begin{bmatrix} 4 & -2 & 0 \\ -2 & 9 & 3 \\ 0 & 3 & 1 \end{bmatrix}$ for variables $(X, Y, Z)$. Then: $\mathrm{Var}(Y) = 9$ (diagonal), $\mathrm{Cov}(X, Y) = -2$ (they move oppositely), $\mathrm{Cov}(X, Z) = 0$ (no linear relation), $\mathrm{Cov}(Y, Z) = 3$ — and the matrix is symmetric, as it must be.

### Example 3: how many numbers?

For $n$ variables the matrix has $n$ variances and $\binom{n}{2}$ distinct covariances: 2 variables → 1 covariance, 3 → 3, 5 → 10, 100 → 4,950. The matrix view keeps this bookkeeping automatic.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

# the ten children from the covariance notebook, all four variables
age = np.array([6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
height = np.array([50, 52, 55, 57, 60, 62, 64, 65, 67, 68])
grades = np.array([5, 7, 8, 3, 1, 1, 6, 10, 2, 7])
naps = np.array([8, 7, 6, 5, 4, 3, 2, 1, 1, 0])

### Building a covariance matrix by hand — and with `np.cov`

In [ ]:
def cov(x, y):
    return ((x - x.mean()) * (y - y.mean())).mean()

variables = {"age": age, "height": height, "grades": grades, "naps": naps}
names = list(variables)

sigma = pd.DataFrame([[cov(variables[a], variables[b]) for b in names]
                      for a in names], index=names, columns=names)
print("covariance matrix of the four child variables (population, /n):")
print(sigma.round(2))

data = np.vstack([age, height, grades, naps])
print(f"\nnp.cov(..., ddof=0) matches: "
      f"{np.allclose(np.cov(data, ddof=0), sigma.values)}")
print("(np.cov's default ddof=1 divides by n-1 instead -- same structure,")
print("slightly larger numbers.)")

### The built-in structure: symmetry and variances on the diagonal

In [ ]:
print(f"symmetric (Sigma == Sigma^T)?  {np.allclose(sigma.values, sigma.values.T)}")
print(f"diagonal: {np.diag(sigma.values).round(2)}")
print(f"variances: {[round(v.var(), 2) for v in variables.values()]}")
print()
print("Off-diagonal highlights: Cov(age, height) = 17 (>0), Cov(age, naps) = -7.45")
print("(<0), Cov(age, grades) = 0.1 (~0) -- the whole Lesson 4 story in one matrix.")

### The games' matrices, from their joint distributions

In [ ]:
def game_sigma(outcomes):
    p = np.array([o[0] for o in outcomes])
    x = np.array([o[1] for o in outcomes], dtype=float)
    y = np.array([o[2] for o in outcomes], dtype=float)
    mx, my = np.sum(p * x), np.sum(p * y)
    vx = np.sum(p * (x - mx)**2)
    vy = np.sum(p * (y - my)**2)
    c = np.sum(p * (x - mx) * (y - my))
    return np.array([[vx, c], [c, vy]])

game1 = [(1/2, 1, 1), (1/2, -1, -1)]
game2 = [(1/2, 1, -1), (1/2, -1, 1)]
game3 = [(1/4, 1, 1), (1/4, 1, -1), (1/4, -1, 1), (1/4, -1, -1)]

for name, g in [("game 1", game1), ("game 2", game2), ("game 3", game3)]:
    print(f"{name}:\n{game_sigma(g)}\n")

### The call-center matrix

In [ ]:
rng = np.random.default_rng(42)

n = 400_000
quick = rng.random(n) < 0.5
wait = np.clip(np.where(quick, rng.normal(2.4, 1.6, n), rng.normal(7.6, 1.6, n)), 0, 10)
rate = np.clip(np.where(quick, rng.normal(8.0, 1.7, n), rng.normal(2.6, 1.7, n)), 0, 10)

sigma_cc = np.cov(wait, rate, ddof=0)
print("call-center covariance matrix (video: [[8.526, -7.878], [-7.878, 10.163]]):")
print(sigma_cc.round(3))

### Five variables: the pattern at scale

A synthetic 5-variable dataset $(V, W, X, Y, Z)$ with planted relationships — the $5\times5$ matrix collects 5 variances and 10 covariances automatically.

In [ ]:
rng = np.random.default_rng(3)

n = 50_000
v = rng.normal(0, 1, n)
w = 0.8 * v + rng.normal(0, 0.6, n)        # strongly follows V
x = -0.5 * v + rng.normal(0, 1, n)         # opposes V
y = rng.normal(0, 2, n)                    # unrelated, big spread
z = 0.5 * w + 0.5 * y + rng.normal(0, 0.4, n)

data5 = np.vstack([v, w, x, y, z])
sigma5 = pd.DataFrame(np.cov(data5, ddof=0), index=list("VWXYZ"),
                      columns=list("VWXYZ"))
print(sigma5.round(2))
print(f"\n5 variances on the diagonal + {5 * 4 // 2} distinct covariances "
      f"(each shown twice by symmetry).")

## 5. Visualizations

### The 4×4 children matrix as an annotated heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
vmax = np.abs(sigma.values).max()
im = ax.imshow(sigma.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(4)); ax.set_xticklabels(names)
ax.set_yticks(range(4)); ax.set_yticklabels(names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{sigma.values[i, j]:.2f}", ha="center", va="center",
                fontsize=11,
                color="white" if abs(sigma.values[i, j]) > vmax * 0.5 else "black")
ax.add_patch(plt.Rectangle((-0.5, -0.5), 4, 4, fill=False, linewidth=0))
for k in range(4):
    ax.add_patch(plt.Rectangle((k - 0.5, k - 0.5), 1, 1, fill=False,
                               edgecolor="black", linewidth=2))
ax.set_title("children data: variances boxed on the diagonal,\ncovariances elsewhere (red +, blue -)")
fig.colorbar(im, label="value")
plt.tight_layout()
plt.show()

### Matrix next to meaning: the pairs plot

Each off-diagonal scatter corresponds to one covariance cell; the diagonal shows each variable alone (its variance). The covariance matrix is this entire figure compressed into numbers.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        ax = axes[i, j]
        if i == j:
            ax.hist(variables[a], bins=8, color="tab:gray", alpha=0.7)
            ax.set_title(f"Var({a}) = {variables[a].var():.2f}", fontsize=10)
        else:
            c = cov(variables[b], variables[a])
            color = "tab:red" if c > 1 else ("tab:blue" if c < -1 else "tab:gray")
            ax.scatter(variables[b], variables[a], s=25, color=color)
            ax.set_title(f"Cov = {c:.2f}", fontsize=10)
        if i == 3:
            ax.set_xlabel(b)
        if j == 0:
            ax.set_ylabel(a)
        ax.tick_params(labelsize=7)
plt.suptitle("The pairs plot IS the covariance matrix, drawn out", y=0.995)
plt.tight_layout()
plt.show()

### The 5×5 matrix heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
vmax5 = np.abs(sigma5.values).max()
im = ax.imshow(sigma5.values, cmap="RdBu_r", vmin=-vmax5, vmax=vmax5)
ax.set_xticks(range(5)); ax.set_xticklabels(sigma5.columns)
ax.set_yticks(range(5)); ax.set_yticklabels(sigma5.index)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{sigma5.values[i, j]:.1f}", ha="center", va="center",
                fontsize=10)
ax.set_title("five variables, one object: $\\Sigma$")
fig.colorbar(im, label="value")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Write down (by hand) the covariance matrix of game 4 from the last notebook ($\mathrm{Var} = \mathrm{Cov} = \tfrac{29}{36}$), then verify with `game_sigma`.

**Exercise 2.** From the children's 4×4 matrix, read off: $\mathrm{Var}(\text{grades})$, $\mathrm{Cov}(\text{height}, \text{naps})$, and $\mathrm{Cov}(\text{naps}, \text{height})$. Why are the last two equal before you even look?

### Intermediate

**Exercise 3.** How many *distinct* numbers does an $n \times n$ covariance matrix contain (diagonal plus one triangle)? Give the formula, evaluate it for $n = 2, 3, 5, 100$, and check against the 4-variable children matrix.

**Exercise 4.** The covariance of height with naps is $-15.5$, and with grades $-0.9$. Verify these from the matrix and interpret both in a sentence, remembering that height is essentially a proxy for age in this data.

### Advanced

**Exercise 5.** A covariance matrix cannot be arbitrary: verify on the children data that every off-diagonal entry satisfies $|\mathrm{Cov}(X_i, X_j)| \le \sigma_i \sigma_j$ (a Cauchy–Schwarz bound). Then try to build a "covariance matrix" $\begin{bmatrix} 1 & 3 \\ 3 & 1 \end{bmatrix}$ from any dataset — why is it impossible?

**Exercise 6.** Covariance matrices power portfolio risk: for two assets with daily-return variances $0.04$ and $0.09$ and covariance $c$, the variance of a 50/50 portfolio is $\tfrac14(0.04) + \tfrac14(0.09) + \tfrac12 c$. Compute the portfolio variance for $c = +0.05, 0, -0.05$ and explain why negative covariance is a diversifier's dream. (This is $\mathrm{Var}(aX + bY) = a^2\mathrm{Var}X + b^2\mathrm{Var}Y + 2ab\,\mathrm{Cov}$ — the two-variable variance rule completed by the covariance term.)

## 7. Solutions

In [ ]:
# Exercise 1: game 4's matrix
game4 = [(1/2, 1, 1), (1/3, -1, -1), (1/6, 0, 0)]
print("Exercise 1:")
print(game_sigma(game4).round(3))
print(f"All four entries are 29/36 = {29/36:.3f}: Y = X always, so variances and")
print("covariance coincide.")

In [ ]:
# Exercise 2: three lookups
print(f"Exercise 2: Var(grades) = {sigma.loc['grades', 'grades']:.2f}")
print(f"Cov(height, naps) = {sigma.loc['height', 'naps']:.2f}")
print(f"Cov(naps, height) = {sigma.loc['naps', 'height']:.2f}")
print("Equal by symmetry: the product (h - mu_h)(n - mu_n) has no order.")

In [ ]:
# Exercise 3: counting distinct entries
def distinct(n):
    return n * (n + 1) // 2
print("Exercise 3: n(n+1)/2 distinct numbers (n variances + n(n-1)/2 covariances)")
for n_ in [2, 3, 5, 100]:
    print(f"  n = {n_:>3}: {distinct(n_)}")
print(f"children (n=4): {distinct(4)} distinct values in the 16-cell matrix.")

In [ ]:
# Exercise 4: reading two relationships
print(f"Exercise 4: Cov(height, naps)  = {sigma.loc['height', 'naps']:.2f} < 0:")
print("taller children (i.e. older ones) nap less -- height inherits age's")
print("negative relation with naps.")
print(f"Cov(height, grades) = {sigma.loc['height', 'grades']:.2f} ~ 0: height,")
print("like age, carries no information about grades.")

In [ ]:
# Exercise 5: the Cauchy-Schwarz constraint
stds = {n_: np.sqrt(variables[n_].var()) for n_ in names}
print("Exercise 5: |Cov| <= sigma_i * sigma_j for every pair:")
ok = True
for i, a in enumerate(names):
    for b in names[i + 1:]:
        bound = stds[a] * stds[b]
        c = abs(sigma.loc[a, b])
        ok &= c <= bound + 1e-9
        print(f"  |Cov({a}, {b})| = {c:>6.2f} <= {bound:>6.2f}")
print(f"all satisfied: {ok}")
print()
print("[[1, 3], [3, 1]] would need |Cov| = 3 > 1 = sigma_x * sigma_y --")
print("impossible: no data can co-vary more strongly than its own spreads allow.")

In [ ]:
# Exercise 6: portfolio variance and diversification
var_a, var_b = 0.04, 0.09
print("Exercise 6:")
for c in [0.05, 0.0, -0.05]:
    port = 0.25 * var_a + 0.25 * var_b + 0.5 * c
    print(f"  Cov = {c:+.2f}: portfolio variance = {port:.4f}")
print("Negative covariance means one asset tends to rise when the other falls,")
print("cancelling swings: the 50/50 mix is LESS risky than either asset alone")
print("(0.0075 < 0.04). The covariance matrix is the whole input to this math.")

## 8. Conclusion

- The **covariance matrix** $\Sigma$ organizes all second-order information of a multivariate dataset: $\Sigma_{ij} = \mathrm{Cov}(X_i, X_j)$, so **variances fill the main diagonal** and covariances fill the rest.
- Small examples: call center $\begin{bmatrix} 8.526 & -7.878 \\ -7.878 & 10.163 \end{bmatrix}$; game 1 $\begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}$ vs game 2 $\begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$ — same diagonal, telling corners.
- Structure for free: symmetry ($\mathrm{Cov}(X,Y) = \mathrm{Cov}(Y,X)$), diagonal = variances, and a Cauchy–Schwarz constraint $|\Sigma_{ij}| \le \sigma_i\sigma_j$ limiting what matrices are possible.
- For $n$ variables: $n$ variances plus $\binom{n}{2}$ covariances — $\Sigma$ scales the bookkeeping to 5, 100, or a million variables, which is why it is everywhere in statistics and ML (multivariate normals, PCA, portfolio risk, `np.cov` in one call).
- A covariance's *magnitude* still depends on units — the standardized version of this whole story is the **correlation coefficient**, up next.